In [1]:
!pip install transformers sentencepiece scikit-learn --quiet

In [2]:
import os
import torch
import warnings
import logging
import numpy as np
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    BertTokenizer, BertForSequenceClassification
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive
import time
from tqdm import tqdm

In [3]:
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger().setLevel(logging.ERROR)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

drive.mount('/content/drive', force_remount=True)

drive_path = '/content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度'
os.chdir(drive_path)

Mounted at /content/drive


In [4]:
device = 0 if torch.cuda.is_available() else -1
print("Device set to", "GPU" if device == 0 else "CPU")

Device set to GPU


In [5]:
nllb_model_name = "facebook/nllb-200-3.3B"
nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name)

translator = pipeline(
    "translation",
    model=nllb_model,
    tokenizer = nllb_tokenizer,
    src_lang = "zho_Hans",
    tgt_lang = "eng_Latn",
    device=device,
    truncation = True,
    max_length = 512
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

pytorch_model-00002-of-00003.bin:   0%|          | 0.00/8.55G [00:00<?, ?B/s]

pytorch_model-00003-of-00003.bin:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [6]:
finbert_model = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(finbert_model)
model = BertForSequenceClassification.from_pretrained(finbert_model)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model = model,
    tokenizer = tokenizer,
    device = device
)

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

In [7]:
equity_reports = pd.read_csv(
    'equity_reports.csv',
    encoding='utf-8',
    parse_dates=['PUBLISH_DATE', 'UPDATE_TIME', 'UPDATE_DATE']
)

In [11]:
sentiment_map = {
    'POSITIVE': '正面',
    'NEUTRAL': '中性',
    'NEGATIVE': '负面'
}

verbose = True

In [15]:
start_time = time.time()

for i, row in equity_reports.head(20).iterrows():
    if verbose:
        print(f"\n=== Row {i + 1} ===")

    title_cn = row['TITLE']
    summary_cn = row['SUMMARY']

    try:
        if pd.notna(title_cn):
            title_en = translator(title_cn, max_length=512, truncation=True)[0]['translation_text']
            title_sentiment = sentiment_pipeline(title_en)[0]
            title_label = sentiment_map.get(title_sentiment['label'].upper(), '[未知]')
            if verbose:
                print(f"TITLE: {title_cn[:100]}{'...' if len(title_cn) > 100 else ''}")
                print(f"  ↳ {title_en[:100]}{'...' if len(title_en) > 100 else ''}")
                print(f"  → 情绪: {title_label} ({title_sentiment['score'] * 100:.1f}%)")
        else:
            if verbose:
                print("TITLE: [缺失]")

        if pd.notna(summary_cn):
            summary_en = translator(summary_cn, max_length=512, truncation=True)[0]['translation_text']
            summary_sentiment = sentiment_pipeline(summary_en)[0]
            summary_label = sentiment_map.get(summary_sentiment['label'].upper(), '[未知]')
            if verbose:
                print(f"SUMMARY: {summary_cn[:100]}{'...' if len(summary_cn) > 100 else ''}")
                print(f"  ↳ {summary_en[:100]}{'...' if len(summary_en) > 100 else ''}")
                print(f"  → 情绪: {summary_label} ({summary_sentiment['score'] * 100:.1f}%)")
        else:
            if verbose:
                print("SUMMARY: [缺失]")

    except Exception as e:
        print(f"Error on row {i}: {e}")

end_time = time.time()


=== Row 1 ===
TITLE: 诺诚健华-B（09969）：奥布替尼两项淋巴瘤适应症获批，纳入港股通期待估值修复
  ↳ Nochin Jinhwa-B ((09969)): Obtini's two lymphoma adaptations approved for inclusion in the Hong Kong...
  → 情绪: 中性 (100.0%)
SUMMARY:     投资要点事件：近日，（1）国家药监局批准公司的1类创新药奥布替尼（商品名：宜诺凯）上市，用于治疗既往至少接受过一种治疗的成人套细胞淋巴瘤（MCL）患者和既往至少接受过一种治疗的成人慢性淋巴细胞...
  ↳ Sullivan & Sullivan, one of the largest pharmaceutical companies in the world, has a projected annua...
  → 情绪: 正面 (92.6%)

=== Row 2 ===
TITLE: 金山办公（688111）公司深度报告之二：循成长之迹，探WPS发展远景
  ↳ The second in-depth report of the company: the trajectory of growth, exploring the prospects for the...
  → 情绪: 中性 (99.9%)
SUMMARY: 战略明确，长期成长路径清晰，维持“买入”评级无论是中端期，还是长期视角看，公司均表现出优异的成长性。考虑公司将持续加大研发投入，我们调整公司2020-2022年归母净利润为8.63、12.25、16....
  ↳ The company will continue to increase its R&D investment, we adjusted the company's net profit in 20...
  → 情绪: 正面 (100.0%)

=== Row 3 ===
TITLE: 共创草坪（605099）：股权激励划定明确目标，人造草坪龙头再添增长动力
  ↳ 605099): Shareholder incentives set clear goals, artificial turf faucets add to 

In [10]:
elapsed = end_time - start_time
print(f"\nProcessed 100 rows in {elapsed:.2f} seconds")
print(f"Average time per row: {elapsed / 100:.2f} seconds")


Processed 100 rows in 803.15 seconds
Average time per row: 8.03 seconds


In [17]:
summary_cn = (
    "公司一季度营收同比增长 5.2%，低于市场预期的 7% 增速。"
    "毛利率下滑 1.0 个百分点，主要由于原材料价格上涨。"
    "虽然管理费用率保持稳健，但研发投入持续扩大，短期利润承压。"
    "考虑到行业整体需求放缓与竞争加剧，我们认为公司短期需谨慎观察，维持“中性”评级。"
)

summary_en = translator(summary_cn, truncation=True, max_length=512)[0]['translation_text']
sentiment = sentiment_pipeline(summary_en)[0]
label = sentiment_map.get(sentiment['label'].upper(), '[未知]')

print("SUMMARY (CN):")
print(summary_cn)
print("\nTRANSLATED (EN):")
print(summary_en)
print(f"\n→ 情绪: {label} ({sentiment['score'] * 100:.1f}%)")

SUMMARY (CN):
公司一季度营收同比增长 5.2%，低于市场预期的 7% 增速。毛利率下滑 1.0 个百分点，主要由于原材料价格上涨。虽然管理费用率保持稳健，但研发投入持续扩大，短期利润承压。考虑到行业整体需求放缓与竞争加剧，我们认为公司短期需谨慎观察，维持“中性”评级。

TRANSLATED (EN):
The company's revenue in the first quarter grew by 5.2% year-on-year, below the 7% growth rate expected by the market. The gross profit margin decreased by 1.0 percentage points, mainly due to higher raw material prices. While the management expense ratio remained solid, R&D investment continued to expand, and short-term profits were under pressure. Considering the slowdown in overall industry demand and increased competition, we believe the company needs to be cautious in the short-term to maintain a neutral rating.

→ 情绪: 负面 (100.0%)
